# Training (PyTorch) - Hybrid Heatmap + Count Head Model

**Notebook role:** Phase 5 (PyTorch port). Trains all three model variants and saves checkpoints.

**Strategy:**
1. Train the *large* variant from scratch on the train split (16 windows).
2. Train *small* and *medium* with knowledge distillation from the trained *large* model.
3. Auto-resume: if a `.pt` checkpoint already exists for a variant, skip training and reload it.

**Inputs**
- `other_files/preprocessing.py`, `heatmap.py`, `model_torch.py`, `convert_to_litert.py`
- `other_files/normalization_stats.npz` (from notebook 02)
- `data/multi-person-localization/data/window_*.npz`

**Outputs**
- `other_files/checkpoints/{small,medium,large}.pt` -- best per val_loss
- `other_files/training_report.json` -- machine-readable metrics
- `other_files/training_report.pdf` -- human-readable training summary
- `other_files/figs_eda/training_*.png` -- four figures

**How to use**
- Set `EPOCHS` and `STEPS_PER_EPOCH` in the config cell to control wall-clock time.
- For a local CPU smoke test: `EPOCHS=2, STEPS_PER_EPOCH=20` (~5-10 min total).
- For a Colab GPU run: `EPOCHS=30, STEPS_PER_EPOCH=200` (~1 hour total).
- For a thorough run: `EPOCHS=80, STEPS_PER_EPOCH=300` (~3 hours on GPU).
- Re-running this notebook is safe; finished variants are skipped.
- Set `FORCE_RETRAIN=True` to ignore checkpoints and retrain from scratch.


## 0. Setup & configuration

In [ ]:
import sys, os, json, time, gc
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import IterableDataset, DataLoader

%matplotlib inline
plt.rcParams["figure.dpi"] = 100
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

ROOT = Path("..").resolve()
OTHER_FILES = ROOT / "other_files"
sys.path.insert(0, str(OTHER_FILES))

DATA_DIR    = ROOT / "data" / "multi-person-localization"
WINDOWS_DIR = DATA_DIR / "data"
FIG_DIR     = OTHER_FILES / "figs_eda"
CKPT_DIR    = OTHER_FILES / "checkpoints"
FIG_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch:    {torch.__version__}")
print(f"Device:     {DEVICE}")


### Configuration -- edit these for local vs. Colab

In [ ]:
# ============================================================
# TRAINING CONFIG -- adjust these based on environment
# ============================================================
# For a local CPU smoke test (5-10 min):
#     EPOCHS = 2; STEPS_PER_EPOCH = 20; BATCH_SIZE = 16
#
# For a Colab GPU run (~1 hour total):
#     EPOCHS = 30; STEPS_PER_EPOCH = 200; BATCH_SIZE = 32
#
# For a thorough run (~3+ hours):
#     EPOCHS = 80; STEPS_PER_EPOCH = 300; BATCH_SIZE = 32
EPOCHS          = 2
STEPS_PER_EPOCH = 20
VAL_STEPS       = 10
BATCH_SIZE      = 16

LEARNING_RATE   = 1e-3
LAMBDA_COUNT    = 0.1     # weight on the count loss in the total loss
DISTILL_ALPHA   = 0.5     # blending weight: 0=hard targets only, 1=teacher only
SEED            = 42

FORCE_RETRAIN   = False    # set True to wipe checkpoints and start over

# Dataset config (matches earlier notebooks)
TRAIN_IDS = [0, 1, 3, 5, 6, 7, 9, 11, 12, 13, 16, 17, 18, 19, 22, 23]
VAL_IDS   = [4, 8, 14, 20]
TEST_IDS  = [2, 10, 15, 21]

print(f"Epochs:           {EPOCHS}")
print(f"Steps per epoch:  {STEPS_PER_EPOCH}")
print(f"Batch size:       {BATCH_SIZE}")
print(f"Total train steps:{EPOCHS * STEPS_PER_EPOCH}")
print(f"Frames seen:      ~{EPOCHS * STEPS_PER_EPOCH * BATCH_SIZE:,}")
print(f"Force retrain:    {FORCE_RETRAIN}")


In [ ]:
# Set seeds for reproducibility
import random
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Import modules
from preprocessing import Preprocessor
from heatmap import (
    xy_to_heatmap, heatmap_to_xy, count_target,
    GRID_H, GRID_W, COUNT_CLASSES,
)
from model_torch import (
    UWBLocalizer, build_model, MODEL_VARIANTS,
    T_CONTEXT, N_RADARS, N_ANT, N_BINS, N_IQ,
)
from convert_to_litert import convert_float32, convert_int8, inspect_tflite

# Load the preprocessor fitted in notebook 02
pp = Preprocessor.load(OTHER_FILES / "normalization_stats.npz")
print(f"Preprocessor loaded: bin_lo={pp.bin_lo}, bin_hi={pp.bin_hi}")
print(f"Normalisation: mean shape {pp.norm.mean.shape}, std shape {pp.norm.std.shape}")

# Window file paths
window_files = sorted(WINDOWS_DIR.glob("window_*.npz"))
train_paths = [window_files[i] for i in TRAIN_IDS]
val_paths   = [window_files[i] for i in VAL_IDS]
test_paths  = [window_files[i] for i in TEST_IDS]
print(f"\nTrain: {len(train_paths)} windows, "
      f"Val: {len(val_paths)} windows, Test: {len(test_paths)} windows")


## 1. Streaming dataset

The full preprocessed train set ($16 \times 7500$ frames $\times 8$ context $\times 30240$ floats $\approx 110$ GB) is far too large to cache. We stream lazily: each call to the generator opens one window, applies the preprocessor in-memory once, then yields random `(stacked_cir, heatmap, count)` triples from it.

**Class balancing:** the 5 occupancy classes (0, 1, 2, 3, 4 people) are represented by 2, 6, 5, 4, 7 windows respectively. We sample windows weighted by the inverse number of windows per class so each class contributes equally to the gradient.

PyTorch implementation uses `IterableDataset` for streaming, with multinomial window sampling at each step.


In [ ]:
def get_window_occupancy(window_path):
    """Return the (constant) people count in a window."""
    with np.load(window_path) as f:
        mask = f["people_mask"]
    return int(mask.sum(axis=-1).max())


# Class balancing: count windows per occupancy class
train_class = np.array([get_window_occupancy(p) for p in train_paths])
val_class   = np.array([get_window_occupancy(p) for p in val_paths])
print("Train windows by occupancy class:")
for k in range(COUNT_CLASSES):
    n = (train_class == k).sum()
    print(f"  {k} people: {n} windows")

# Sampling weight = 1 / (number of windows in this class)
class_counts = np.bincount(train_class, minlength=COUNT_CLASSES)
window_weights = 1.0 / np.maximum(class_counts[train_class], 1)
window_weights = window_weights / window_weights.sum()   # probabilities
print(f"\nPer-window sampling probabilities (training):")
for p, w, c in zip(train_paths, window_weights, train_class):
    print(f"  {p.name}  class={c}  P={w:.4f}")


In [ ]:
class WindowDataset(IterableDataset):
    """Streaming dataset: samples frames from one or more .npz windows.

    Behaviour
    ---------
    - On every iteration, picks a window according to ``weights`` (or uniform
      if None), then loads it, preprocesses, and yields random frames from it.
      Repeat indefinitely if ``infinite=True``.
    - Yields tuples ``(stacked_cir, heatmap_target, count_target)`` where:
        stacked_cir  : float32 (T_CONTEXT, R, A, B, IQ)
        heatmap_target : float32 (1, GRID_H, GRID_W) -- channels-first for PyTorch
        count_target   : int64 scalar
    """

    def __init__(self, window_paths, preprocessor,
                 weights=None, infinite=True, seed=42):
        super().__init__()
        self.paths = list(window_paths)
        self.pp = preprocessor
        self.weights = weights
        self.infinite = infinite
        self.seed = seed

    def _stream_window(self, path, rng):
        """Load one window, preprocess, yield frames in a random order."""
        with np.load(path) as f:
            cir   = f["radar_cir_iq"].astype(np.float32)
            xy    = f["people_xy"]
            mask  = f["people_mask"]

        # Preprocess once
        cir_pp = self.pp.transform(cir)            # (T, R, A, 105, IQ)
        T = cir_pp.shape[0]

        # Build all heatmap targets in one go
        heatmaps = xy_to_heatmap(xy, mask)         # (T, 36, 24)
        heatmaps = heatmaps[:, None, :, :]         # (T, 1, 36, 24) -- channels-first
        counts   = count_target(mask)              # (T,)

        # Pre-pad for causal frame stacks: replicate cir_pp[0] (T_CONTEXT-1) times
        pad = np.repeat(cir_pp[:1], T_CONTEXT - 1, axis=0)
        padded = np.concatenate([pad, cir_pp], axis=0)

        # Skip warm-up frames (first 50)
        valid_t = np.arange(50, T)
        order = rng.permutation(valid_t)
        for t in order:
            stack = padded[t:t + T_CONTEXT]        # (T_CONTEXT, R, A, 105, IQ)
            yield (
                torch.from_numpy(stack.copy()),
                torch.from_numpy(heatmaps[t].copy()).float(),
                torch.tensor(counts[t], dtype=torch.long),
            )

    def __iter__(self):
        # Set up per-worker RNG so DataLoader workers don't all see the same data
        worker_info = torch.utils.data.get_worker_info()
        worker_id = worker_info.id if worker_info else 0
        rng = np.random.RandomState(self.seed + worker_id * 1000)

        while True:
            if self.weights is not None:
                idx = int(rng.choice(len(self.paths), p=self.weights))
            else:
                idx = int(rng.choice(len(self.paths)))
            yield from self._stream_window(self.paths[idx], rng)
            if not self.infinite:
                break


# Build datasets
train_ds = WindowDataset(train_paths, pp, weights=window_weights,
                         infinite=True, seed=SEED)
val_ds   = WindowDataset(val_paths,   pp, weights=None,
                         infinite=True, seed=SEED + 100)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, num_workers=0)
print("Datasets ready.")


## 2. Sanity check: visualise a few batches

Pull 4 batches from the training pipeline, visualise the heatmap target and verify the count distribution looks balanced.


In [ ]:
print("Sampling 4 batches from the training pipeline...")
counts_seen = []
batch_iter = iter(train_loader)
sample_batch = None

t0 = time.time()
for i in range(4):
    cir_batch, hm_batch, cnt_batch = next(batch_iter)
    counts_seen.extend(cnt_batch.numpy().tolist())
    if i == 0:
        sample_batch = (cir_batch.numpy(), hm_batch.numpy(), cnt_batch.numpy())
    print(f"  Batch {i+1}: cir {tuple(cir_batch.shape)}, "
          f"hm {tuple(hm_batch.shape)}, count {tuple(cnt_batch.shape)}, "
          f"hm range [{float(hm_batch.min()):.3f}, {float(hm_batch.max()):.3f}]")
elapsed = time.time() - t0
n_frames = 4 * BATCH_SIZE
print(f"\n4 batches ({n_frames} frames) took {elapsed:.2f} s = {n_frames/elapsed:.1f} frames/s")

import collections
ctr = collections.Counter(counts_seen)
print(f"\nCount distribution across {len(counts_seen)} sampled frames "
      f"(target: equal across 0-4):")
for k in range(COUNT_CLASSES):
    print(f"  {k} people: {ctr.get(k, 0):>3d} frames ({ctr.get(k, 0)/len(counts_seen)*100:.1f}%)")


In [ ]:
# Visualise the first batch's heatmap targets + counts
cir_b, hm_b, cnt_b = sample_batch
N_SHOW = min(6, len(cnt_b))

fig, axes = plt.subplots(1, N_SHOW, figsize=(N_SHOW*2.5, 4))
if N_SHOW == 1:
    axes = [axes]
for ax, h, c in zip(axes, hm_b[:N_SHOW], cnt_b[:N_SHOW]):
    # h is (1, 36, 24) -- show channel 0
    ax.imshow(h[0], cmap="hot", origin="lower", vmin=0, vmax=1)
    ax.set_title(f"count = {int(c)}", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle(f"First {N_SHOW} heatmap targets in a sampled batch", y=1.02, fontsize=10)
plt.tight_layout()
plt.savefig(FIG_DIR / "training_sample_batch.png", bbox_inches="tight")
plt.show()


## 3. Training utilities

Custom training loop with `tqdm` progress bars and best-checkpoint saving.


In [ ]:
!pip install tqdm -q


In [ ]:
from tqdm.auto import tqdm


def train_one_variant(variant_name, train_loader, val_loader,
                      epochs, steps_per_epoch, val_steps,
                      lr=1e-3, lambda_count=0.1,
                      ckpt_path=None, force_retrain=False):
    """Train a single variant from scratch with hard targets only.

    If a checkpoint exists at ckpt_path and force_retrain is False, the
    function loads it and returns immediately (auto-resume).
    Returns (model, history_dict, time_taken).
    """
    if ckpt_path is not None and ckpt_path.exists() and not force_retrain:
        print(f"  Loading existing checkpoint: {ckpt_path}")
        model = build_model(variant_name).to(DEVICE)
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
        return model, None, 0.0

    model = build_model(variant_name).to(DEVICE)
    bce = nn.BCELoss()
    ce  = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=5, min_lr=1e-5,
    )

    history = {
        "train_loss": [], "train_hm_loss": [], "train_cnt_loss": [],
        "train_cnt_acc": [],
        "val_loss":   [], "val_hm_loss":   [], "val_cnt_loss":   [],
        "val_cnt_acc":   [],
        "lr": [],
    }
    best_val_loss = float("inf")

    print(f"  Training {variant_name}: {model.num_params:,} params on {DEVICE}")
    t0 = time.time()

    train_iter = iter(train_loader)
    val_iter   = iter(val_loader)

    for epoch in range(epochs):
        # ---- Training phase ----
        model.train()
        tr_loss = tr_hm = tr_cnt = tr_acc = 0.0
        n = 0
        pbar = tqdm(range(steps_per_epoch),
                    desc=f"  Epoch {epoch+1}/{epochs}", leave=False)
        for _ in pbar:
            cir, hm_true, cnt_true = next(train_iter)
            cir, hm_true, cnt_true = cir.to(DEVICE), hm_true.to(DEVICE), cnt_true.to(DEVICE)

            optimizer.zero_grad()
            hm_pred, cnt_pred = model(cir)
            hm_loss  = bce(hm_pred, hm_true)
            cnt_loss = ce(cnt_pred, cnt_true)
            loss = hm_loss + lambda_count * cnt_loss
            loss.backward()
            optimizer.step()

            with torch.no_grad():
                cnt_correct = (cnt_pred.argmax(1) == cnt_true).float().mean()
            tr_loss += loss.item();  tr_hm += hm_loss.item();  tr_cnt += cnt_loss.item()
            tr_acc += cnt_correct.item()
            n += 1
            pbar.set_postfix(loss=f"{loss.item():.4f}",
                             cnt_acc=f"{cnt_correct.item():.3f}")

        tr_loss /= n; tr_hm /= n; tr_cnt /= n; tr_acc /= n

        # ---- Validation phase ----
        model.eval()
        v_loss = v_hm = v_cnt = v_acc = 0.0
        nv = 0
        with torch.no_grad():
            for _ in range(val_steps):
                cir, hm_true, cnt_true = next(val_iter)
                cir, hm_true, cnt_true = cir.to(DEVICE), hm_true.to(DEVICE), cnt_true.to(DEVICE)
                hm_pred, cnt_pred = model(cir)
                hm_loss  = bce(hm_pred, hm_true)
                cnt_loss = ce(cnt_pred, cnt_true)
                loss = hm_loss + lambda_count * cnt_loss
                cnt_correct = (cnt_pred.argmax(1) == cnt_true).float().mean()
                v_loss += loss.item(); v_hm += hm_loss.item(); v_cnt += cnt_loss.item()
                v_acc += cnt_correct.item()
                nv += 1
        v_loss /= nv; v_hm /= nv; v_cnt /= nv; v_acc /= nv

        # Step scheduler & log
        scheduler.step(v_loss)
        cur_lr = optimizer.param_groups[0]["lr"]

        history["train_loss"].append(tr_loss)
        history["train_hm_loss"].append(tr_hm)
        history["train_cnt_loss"].append(tr_cnt)
        history["train_cnt_acc"].append(tr_acc)
        history["val_loss"].append(v_loss)
        history["val_hm_loss"].append(v_hm)
        history["val_cnt_loss"].append(v_cnt)
        history["val_cnt_acc"].append(v_acc)
        history["lr"].append(cur_lr)

        msg = (f"  Epoch {epoch+1:>3d}/{epochs}  "
               f"train_loss={tr_loss:.4f} cnt_acc={tr_acc:.3f}  "
               f"val_loss={v_loss:.4f} val_cnt_acc={v_acc:.3f}  lr={cur_lr:.1e}")
        if v_loss < best_val_loss:
            best_val_loss = v_loss
            if ckpt_path is not None:
                torch.save(model.state_dict(), ckpt_path)
            msg += "  [saved]"
        print(msg)

    dt = time.time() - t0
    print(f"  Done in {dt:.1f} s, best val_loss = {best_val_loss:.4f}")

    # Reload best checkpoint
    if ckpt_path is not None and ckpt_path.exists():
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    return model, history, dt


## 4. Train the large variant from scratch

Standard supervised training: BCE on the heatmap, cross-entropy on the count.


In [ ]:
print("=" * 60)
print("TRAINING: large (teacher)")
print("=" * 60)
large_ckpt = CKPT_DIR / "large.pt"

large_model, large_history, large_time = train_one_variant(
    "large", train_loader, val_loader,
    epochs=EPOCHS, steps_per_epoch=STEPS_PER_EPOCH, val_steps=VAL_STEPS,
    lr=LEARNING_RATE, lambda_count=LAMBDA_COUNT,
    ckpt_path=large_ckpt, force_retrain=FORCE_RETRAIN,
)


## 5. Train small and medium with knowledge distillation

The student loss is a blend of hard targets (from the dataset) and soft targets (from the teacher's predictions on the same input):
$$
\mathcal{L}_{\text{student}} = (1 - \alpha) \cdot \mathcal{L}_{\text{hard}}(\hat{H}_s, H, \hat{p}_s, c) + \alpha \cdot \mathcal{L}_{\text{soft}}(\hat{H}_s, \hat{H}_t)
$$

The teacher's heatmap output is more informative than the hard ground-truth heatmap because it encodes uncertainty (smooth peaks, residual signal in nearby cells) that the student can mimic.


In [ ]:
def train_distilled_variant(variant_name, teacher, train_loader, val_loader,
                            epochs, steps_per_epoch, val_steps,
                            lr=1e-3, alpha=0.5, lambda_count=0.1,
                            ckpt_path=None, force_retrain=False):
    """Train a student variant with knowledge distillation from a teacher.

    Total loss:
        (1 - alpha) * (BCE(student_hm, hard_hm)
                       + lambda_count * CE(student_cnt, hard_cnt))
        + alpha     * MSE(student_hm, teacher_hm)
    """
    if ckpt_path is not None and ckpt_path.exists() and not force_retrain:
        print(f"  Loading existing checkpoint: {ckpt_path}")
        model = build_model(variant_name).to(DEVICE)
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
        return model, None, 0.0

    student = build_model(variant_name).to(DEVICE)
    teacher = teacher.to(DEVICE).eval()
    for p in teacher.parameters():
        p.requires_grad_(False)

    bce = nn.BCELoss()
    ce  = nn.CrossEntropyLoss()
    mse = nn.MSELoss()
    optimizer = torch.optim.Adam(student.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=5, min_lr=1e-5,
    )

    history = {
        "train_loss": [], "train_hard": [], "train_soft": [], "train_cnt_acc": [],
        "val_loss":   [], "val_cnt_acc":   [], "lr": [],
    }
    best_val_loss = float("inf")

    print(f"  Training {variant_name}: {student.num_params:,} params (alpha={alpha}) on {DEVICE}")
    t0 = time.time()

    train_iter = iter(train_loader)
    val_iter   = iter(val_loader)

    for epoch in range(epochs):
        # ---- Training ----
        student.train()
        tot_loss = tot_hard = tot_soft = tot_acc = 0.0
        n = 0
        pbar = tqdm(range(steps_per_epoch),
                    desc=f"  Epoch {epoch+1}/{epochs}", leave=False)
        for _ in pbar:
            cir, hm_true, cnt_true = next(train_iter)
            cir, hm_true, cnt_true = cir.to(DEVICE), hm_true.to(DEVICE), cnt_true.to(DEVICE)

            with torch.no_grad():
                t_hm, _ = teacher(cir)

            optimizer.zero_grad()
            s_hm, s_cnt = student(cir)
            hard_loss = bce(s_hm, hm_true) + lambda_count * ce(s_cnt, cnt_true)
            soft_loss = mse(s_hm, t_hm)
            loss = (1.0 - alpha) * hard_loss + alpha * soft_loss
            loss.backward()
            optimizer.step()

            with torch.no_grad():
                cnt_correct = (s_cnt.argmax(1) == cnt_true).float().mean()
            tot_loss += loss.item(); tot_hard += hard_loss.item(); tot_soft += soft_loss.item()
            tot_acc += cnt_correct.item()
            n += 1
            pbar.set_postfix(loss=f"{loss.item():.4f}",
                             hard=f"{hard_loss.item():.4f}",
                             soft=f"{soft_loss.item():.4f}")

        tot_loss /= n; tot_hard /= n; tot_soft /= n; tot_acc /= n

        # ---- Validation ----
        student.eval()
        v_loss = v_acc = 0.0
        nv = 0
        with torch.no_grad():
            for _ in range(val_steps):
                cir, hm_true, cnt_true = next(val_iter)
                cir, hm_true, cnt_true = cir.to(DEVICE), hm_true.to(DEVICE), cnt_true.to(DEVICE)
                s_hm, s_cnt = student(cir)
                hard_loss = bce(s_hm, hm_true) + lambda_count * ce(s_cnt, cnt_true)
                cnt_correct = (s_cnt.argmax(1) == cnt_true).float().mean()
                v_loss += hard_loss.item(); v_acc += cnt_correct.item()
                nv += 1
        v_loss /= nv; v_acc /= nv

        scheduler.step(v_loss)
        cur_lr = optimizer.param_groups[0]["lr"]

        history["train_loss"].append(tot_loss)
        history["train_hard"].append(tot_hard)
        history["train_soft"].append(tot_soft)
        history["train_cnt_acc"].append(tot_acc)
        history["val_loss"].append(v_loss)
        history["val_cnt_acc"].append(v_acc)
        history["lr"].append(cur_lr)

        msg = (f"  Epoch {epoch+1:>3d}/{epochs}  "
               f"loss={tot_loss:.4f} hard={tot_hard:.4f} soft={tot_soft:.4f}  "
               f"val_loss={v_loss:.4f} val_cnt_acc={v_acc:.3f}  lr={cur_lr:.1e}")
        if v_loss < best_val_loss:
            best_val_loss = v_loss
            if ckpt_path is not None:
                torch.save(student.state_dict(), ckpt_path)
            msg += "  [saved]"
        print(msg)

    dt = time.time() - t0
    print(f"  Done in {dt:.1f} s, best val_loss = {best_val_loss:.4f}")

    # Reload best
    if ckpt_path is not None and ckpt_path.exists():
        student.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    return student, history, dt


In [ ]:
# Train medium with distillation
print("=" * 60)
print("TRAINING: medium (distilled from large)")
print("=" * 60)
medium_ckpt = CKPT_DIR / "medium.pt"
medium_model, medium_history, medium_time = train_distilled_variant(
    "medium", large_model, train_loader, val_loader,
    epochs=EPOCHS, steps_per_epoch=STEPS_PER_EPOCH, val_steps=VAL_STEPS,
    lr=LEARNING_RATE, alpha=DISTILL_ALPHA, lambda_count=LAMBDA_COUNT,
    ckpt_path=medium_ckpt, force_retrain=FORCE_RETRAIN,
)


In [ ]:
# Train small with distillation
print("=" * 60)
print("TRAINING: small (distilled from large)")
print("=" * 60)
small_ckpt = CKPT_DIR / "small.pt"
small_model, small_history, small_time = train_distilled_variant(
    "small", large_model, train_loader, val_loader,
    epochs=EPOCHS, steps_per_epoch=STEPS_PER_EPOCH, val_steps=VAL_STEPS,
    lr=LEARNING_RATE, alpha=DISTILL_ALPHA, lambda_count=LAMBDA_COUNT,
    ckpt_path=small_ckpt, force_retrain=FORCE_RETRAIN,
)

models = {"small": small_model, "medium": medium_model, "large": large_model}
histories = {"small": small_history, "medium": medium_history, "large": large_history}
times = {"small": small_time, "medium": medium_time, "large": large_time}


## 6. Loss curves

Compare training and validation loss across the three variants.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

colors = {"small": "tab:green", "medium": "tab:orange", "large": "tab:red"}

for name, hist in histories.items():
    if hist is None:
        continue   # loaded from checkpoint
    epochs_x = np.arange(1, len(hist.get("train_loss", [])) + 1)

    if "train_loss" in hist:
        axes[0].plot(epochs_x, hist["train_loss"], "-",
                     color=colors[name], label=f"{name} (train)")
    if "val_loss" in hist:
        axes[0].plot(epochs_x, hist["val_loss"], "--",
                     color=colors[name], label=f"{name} (val)")

    if "val_cnt_acc" in hist:
        axes[1].plot(epochs_x, hist["val_cnt_acc"], "-",
                     color=colors[name], label=f"{name}")

axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title("Training and validation loss")
axes[0].legend(loc="upper right", fontsize=8)
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Count head accuracy (validation)")
axes[1].set_title("Count head accuracy (val)")
axes[1].legend(loc="lower right", fontsize=8)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.savefig(FIG_DIR / "training_loss_curves.png", bbox_inches="tight")
plt.show()


## 7. Quick test-set evaluation

Run each model on a small slice of the test set and compute mean localisation error per occupancy class.


In [ ]:
def evaluate_on_test(model, test_paths, preprocessor, n_frames_per_window=100):
    """Run inference on test windows, compute mean localisation error.

    Returns dict with per-class errors in cm.
    """
    model.eval()
    all_errs_by_class = {k: [] for k in range(COUNT_CLASSES)}
    count_correct = 0
    count_total = 0

    for path in test_paths:
        with np.load(path) as f:
            cir = f["radar_cir_iq"].astype(np.float32)
            xy_gt = f["people_xy"]
            mask_gt = f["people_mask"]

        cir_pp = preprocessor.transform(cir)
        T = cir_pp.shape[0]
        pad = np.repeat(cir_pp[:1], T_CONTEXT - 1, axis=0)
        padded = np.concatenate([pad, cir_pp], axis=0)

        valid_t = np.arange(50, T)
        sample_t = valid_t[np.linspace(0, len(valid_t) - 1, n_frames_per_window, dtype=int)]

        for t in sample_t:
            stack = padded[t:t + T_CONTEXT][None]   # (1, ...)
            with torch.no_grad():
                stack_t = torch.from_numpy(stack).float().to(DEVICE)
                hm, cnt_p = model(stack_t)
            hm = hm.cpu().numpy()[0, 0]            # (36, 24)
            cnt_p = cnt_p.cpu().numpy()[0]

            k_pred = int(np.argmax(cnt_p))
            k_true = int(count_target(mask_gt[t:t+1])[0])
            count_total += 1
            if k_pred == k_true:
                count_correct += 1
            if k_true == 0:
                continue

            xy_pred = heatmap_to_xy(hm, k=k_true)   # use TRUE k for localization eval
            xy_true = xy_gt[t][mask_gt[t].astype(bool)]

            # Greedy match
            used = set()
            for true_xy in xy_true:
                if len(xy_pred) == 0:
                    continue
                dists = np.linalg.norm(xy_pred - true_xy, axis=1)
                for j in np.argsort(dists):
                    if int(j) not in used:
                        used.add(int(j))
                        all_errs_by_class[k_true].append(dists[j] * 100)
                        break

    out = {
        "count_acc": count_correct / max(1, count_total),
        "n_eval": count_total,
        "errors_by_class_cm": {k: errs for k, errs in all_errs_by_class.items() if errs},
    }
    all_errs = [e for errs in all_errs_by_class.values() for e in errs]
    if all_errs:
        out["mean_err_cm"]   = float(np.mean(all_errs))
        out["median_err_cm"] = float(np.median(all_errs))
        out["p90_err_cm"]    = float(np.percentile(all_errs, 90))
    else:
        out["mean_err_cm"] = out["median_err_cm"] = out["p90_err_cm"] = float("nan")
    return out


print("Evaluating all variants on test set...")
N_FRAMES_EVAL = 100

eval_results = {}
for name, m in models.items():
    print(f"  {name}: ", end="", flush=True)
    t0 = time.time()
    res = evaluate_on_test(m, test_paths, pp, n_frames_per_window=N_FRAMES_EVAL)
    dt = time.time() - t0
    eval_results[name] = res
    print(f"mean_err {res['mean_err_cm']:.1f} cm, "
          f"median {res['median_err_cm']:.1f} cm, "
          f"p90 {res['p90_err_cm']:.1f} cm, "
          f"count_acc {res['count_acc']*100:.1f}% "
          f"({dt:.1f} s)")


In [ ]:
# Plot error comparison
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

names = list(eval_results.keys())
mean_errs = [eval_results[n]["mean_err_cm"] for n in names]
med_errs  = [eval_results[n]["median_err_cm"] for n in names]
p90_errs  = [eval_results[n]["p90_err_cm"] for n in names]
cnt_accs  = [eval_results[n]["count_acc"] * 100 for n in names]

x = np.arange(len(names))
w = 0.27
axes[0].bar(x - w, mean_errs, w, label="Mean", color="steelblue")
axes[0].bar(x,     med_errs,  w, label="Median", color="seagreen")
axes[0].bar(x + w, p90_errs,  w, label="p90", color="firebrick")
axes[0].set_xticks(x); axes[0].set_xticklabels(names)
axes[0].set_ylabel("Localisation error (cm)")
axes[0].set_title(f"Test-set localisation error ({N_FRAMES_EVAL} frames/window)")
axes[0].legend()
axes[0].grid(axis="y", alpha=0.3)
for i, (m, md_, p) in enumerate(zip(mean_errs, med_errs, p90_errs)):
    axes[0].text(i - w, m + 1, f"{m:.0f}", ha="center", fontsize=8)
    axes[0].text(i,     md_ + 1, f"{md_:.0f}", ha="center", fontsize=8)
    axes[0].text(i + w, p + 1, f"{p:.0f}", ha="center", fontsize=8)

axes[1].bar(x, cnt_accs, color=["seagreen", "darkorange", "firebrick"])
axes[1].set_xticks(x); axes[1].set_xticklabels(names)
axes[1].set_ylabel("Count head accuracy (%)")
axes[1].set_title("Count head accuracy on test set")
axes[1].set_ylim(0, 105)
axes[1].grid(axis="y", alpha=0.3)
for i, c in enumerate(cnt_accs):
    axes[1].text(i, c + 1.5, f"{c:.1f}%", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig(FIG_DIR / "training_eval_summary.png", bbox_inches="tight")
plt.show()


## 8. Sample predictions vs. ground truth

Visualise model output on a single test frame for each variant.


In [ ]:
# Pick a 4-people frame from the test set: w10 has 4 people
w_idx = 10
test_path = window_files[w_idx]
sample_t = 3000

with np.load(test_path) as f:
    cir = f["radar_cir_iq"].astype(np.float32)
    xy_gt = f["people_xy"][sample_t]
    mask_gt = f["people_mask"][sample_t]

cir_pp = pp.transform(cir)
pad = np.repeat(cir_pp[:1], T_CONTEXT - 1, axis=0)
padded = np.concatenate([pad, cir_pp], axis=0)
stack = padded[sample_t:sample_t + T_CONTEXT][None]
stack_t = torch.from_numpy(stack).float().to(DEVICE)

# Generate target heatmap for visualisation
gt_heatmap = xy_to_heatmap(xy_gt, mask_gt)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
extent = (0, 4.8, 0, 7.2)
axes[0].imshow(gt_heatmap, cmap="hot", origin="lower", extent=extent, vmin=0, vmax=1)
axes[0].set_title(f"Ground truth\nw{w_idx:02d}, t={sample_t}, k={int(mask_gt.sum())}")

valid = mask_gt.astype(bool)
for ax in axes:
    ax.scatter(xy_gt[valid, 0], xy_gt[valid, 1], marker="o", s=80,
               facecolors="none", edgecolors="cyan", linewidths=1.5)
    ax.set_xlim(0, 4.8); ax.set_ylim(0, 7.2)
    ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)")

for i, name in enumerate(["small", "medium", "large"]):
    with torch.no_grad():
        hm, cnt = models[name](stack_t)
    hm  = hm.cpu().numpy()[0, 0]
    cnt = cnt.cpu().numpy()[0]
    k_pred = int(np.argmax(cnt))
    axes[i + 1].imshow(hm, cmap="hot", origin="lower", extent=extent, vmin=0, vmax=1)
    axes[i + 1].set_title(f"{name}\npred count = {k_pred}, hm range [{hm.min():.2f}, {hm.max():.2f}]")

plt.tight_layout()
plt.savefig(FIG_DIR / "training_sample_predictions.png", bbox_inches="tight")
plt.show()


## 9. Save training report

Write `training_report.json` (machine-readable) and `training_report.pdf` (human-readable).


In [ ]:
# Build the report dict
report = {
    "generated_at": datetime.now().isoformat(),
    "framework": "PyTorch",
    "device": str(DEVICE),
    "config": {
        "epochs": EPOCHS,
        "steps_per_epoch": STEPS_PER_EPOCH,
        "val_steps": VAL_STEPS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "lambda_count": LAMBDA_COUNT,
        "distill_alpha": DISTILL_ALPHA,
        "seed": SEED,
        "train_windows": TRAIN_IDS,
        "val_windows":   VAL_IDS,
        "test_windows":  TEST_IDS,
    },
    "variants": {},
}

for name, m in models.items():
    res = eval_results[name]
    hist = histories[name]
    final = {}
    if hist is not None:
        for k, v in hist.items():
            if isinstance(v, list) and len(v):
                final[k] = float(v[-1])
    report["variants"][name] = {
        "params": int(m.num_params),
        "training_time_s": float(times[name]),
        "final_metrics": final,
        "test_eval": {
            "mean_err_cm":   float(res["mean_err_cm"]),
            "median_err_cm": float(res["median_err_cm"]),
            "p90_err_cm":    float(res["p90_err_cm"]),
            "count_acc":     float(res["count_acc"]),
            "n_eval":        int(res["n_eval"]),
        },
    }

report_path = OTHER_FILES / "training_report.json"
with open(report_path, "w") as f:
    json.dump(report, f, indent=2, default=str)
print(f"Wrote {report_path}  ({report_path.stat().st_size/1024:.1f} KB)")

# Print summary
print("\n" + "=" * 60)
print("TRAINING SUMMARY")
print("=" * 60)
print(f"{'Variant':>8s}  {'Params':>9s}  {'Time':>7s}  {'Mean err':>10s}  "
      f"{'Median':>8s}  {'p90':>7s}  {'Count acc':>10s}")
print("-" * 70)
for name in ["small", "medium", "large"]:
    v = report["variants"][name]
    e = v["test_eval"]
    print(f"{name:>8s}  {v['params']:>9,}  {v['training_time_s']:>5.0f}s   "
          f"{e['mean_err_cm']:>7.1f}cm  {e['median_err_cm']:>5.1f}cm   "
          f"{e['p90_err_cm']:>5.1f}cm   {e['count_acc']*100:>7.1f}%")


In [ ]:
!pip install reportlab pillow -q


In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
    Image as RLImage, PageBreak,
)
from PIL import Image as PILImage

OUT_PDF = OTHER_FILES / "training_report.pdf"

ss = getSampleStyleSheet()
H1   = ParagraphStyle("H1",    parent=ss["Heading1"], fontSize=14, spaceAfter=4, spaceBefore=8,
                               textColor=colors.HexColor("#0b3d91"))
H2   = ParagraphStyle("H2",    parent=ss["Heading2"], fontSize=11, spaceAfter=2, spaceBefore=6,
                               textColor=colors.HexColor("#0b3d91"))
BODY = ParagraphStyle("body",  parent=ss["BodyText"], fontSize=8.5, leading=11)
SMALL= ParagraphStyle("small", parent=ss["BodyText"], fontSize=7.5, leading=9.5)

def tbl(data, col_widths=None, font_size=7.5, header=True):
    t = Table(data, colWidths=col_widths)
    style = [
        ("FONT", (0, 0), (-1, -1), "Helvetica", font_size),
        ("GRID", (0, 0), (-1, -1), 0.25, colors.grey),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("LEFTPADDING", (0, 0), (-1, -1), 3),
        ("RIGHTPADDING", (0, 0), (-1, -1), 3),
        ("TOPPADDING", (0, 0), (-1, -1), 1.5),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 1.5),
    ]
    if header:
        style += [
            ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#e8eef7")),
            ("FONT", (0, 0), (-1, 0), "Helvetica-Bold", font_size),
        ]
    t.setStyle(TableStyle(style))
    return t

def fig(path, width=16*cm):
    p = Path(path)
    if not p.exists():
        return Paragraph(f"<i>(missing: {p.name})</i>", SMALL)
    with PILImage.open(p) as im:
        w, h = im.size
    return RLImage(str(p), width=width, height=width * (h / w))


doc = SimpleDocTemplate(
    str(OUT_PDF), pagesize=A4,
    leftMargin=1.5*cm, rightMargin=1.5*cm,
    topMargin=1.2*cm, bottomMargin=1.2*cm,
    title="Training Report - Multi-Person UWB Localization (PyTorch)",
)
story = []

story.append(Paragraph("Training Report - Multi-Person UWB Localization", H1))
story.append(Paragraph(
    f"Generated {datetime.now():%Y-%m-%d %H:%M} - PyTorch implementation - "
    f"Hybrid heatmap + count head, 3 size variants - "
    f"large trained from scratch; small and medium distilled from large - "
    f"Device: {DEVICE}",
    SMALL,
))
story.append(Spacer(1, 4))

# 1. Configuration
story.append(Paragraph("1. Training configuration", H2))
cfg = report["config"]
config_rows = [
    ["Parameter", "Value"],
    ["Framework",       "PyTorch"],
    ["Device",          str(DEVICE)],
    ["Epochs",          str(cfg["epochs"])],
    ["Steps per epoch", str(cfg["steps_per_epoch"])],
    ["Val steps",       str(cfg["val_steps"])],
    ["Batch size",      str(cfg["batch_size"])],
    ["Total training samples seen", f"{cfg['epochs'] * cfg['steps_per_epoch'] * cfg['batch_size']:,}"],
    ["Learning rate",   f"{cfg['learning_rate']}"],
    ["Lambda count",    f"{cfg['lambda_count']}"],
    ["Distillation alpha", f"{cfg['distill_alpha']}"],
    ["Random seed",     str(cfg["seed"])],
    ["Train windows",   ", ".join(f"w{i:02d}" for i in cfg["train_windows"])],
    ["Val windows",     ", ".join(f"w{i:02d}" for i in cfg["val_windows"])],
    ["Test windows",    ", ".join(f"w{i:02d}" for i in cfg["test_windows"])],
]
story.append(tbl(config_rows, col_widths=[5*cm, 11*cm]))
story.append(Spacer(1, 6))

# 2. Per-variant summary
story.append(Paragraph("2. Per-variant summary", H2))
sum_rows = [["Variant", "Params", "Train time", "Mean err", "Median err", "p90 err", "Count acc"]]
for name in ["small", "medium", "large"]:
    v = report["variants"][name]
    e = v["test_eval"]
    sum_rows.append([
        name,
        f"{v['params']:,}",
        f"{v['training_time_s']:.0f} s",
        f"{e['mean_err_cm']:.1f} cm",
        f"{e['median_err_cm']:.1f} cm",
        f"{e['p90_err_cm']:.1f} cm",
        f"{e['count_acc']*100:.1f}%",
    ])
story.append(tbl(sum_rows, col_widths=[2*cm, 2.2*cm, 2.5*cm, 2.5*cm, 2.5*cm, 2.5*cm, 2.5*cm]))
story.append(Spacer(1, 4))

# 3. Loss curves
story.append(Paragraph("3. Training curves", H2))
story.append(fig(FIG_DIR / "training_loss_curves.png", width=17*cm))

# 4. Test-set evaluation
story.append(PageBreak())
story.append(Paragraph("4. Test-set evaluation", H2))
story.append(Paragraph(
    f"Each model was evaluated on {N_FRAMES_EVAL} evenly-spaced frames per test window "
    f"(test windows: {', '.join(f'w{i:02d}' for i in TEST_IDS)}). Localisation error is "
    f"computed only on frames with at least one person, using the ground-truth count for "
    f"top-k peak extraction.",
    BODY,
))
story.append(fig(FIG_DIR / "training_eval_summary.png", width=17*cm))

# 5. Sample predictions
story.append(Spacer(1, 6))
story.append(Paragraph("5. Sample predictions", H2))
story.append(Paragraph(
    "Side-by-side comparison of ground truth (leftmost) and predicted heatmaps (small, "
    "medium, large) on a single test frame. Cyan circles mark ground-truth positions.",
    BODY,
))
story.append(fig(FIG_DIR / "training_sample_predictions.png", width=17*cm))

# 6. Sample batch
story.append(Spacer(1, 6))
story.append(Paragraph("6. Training batch sanity check", H2))
story.append(Paragraph(
    "Six heatmap targets sampled from the streaming pipeline at training time. "
    "Verifies the inverse-frequency window sampling produces a balanced mix of "
    "occupancy classes and that targets retain peak amplitude 1.0.",
    BODY,
))
story.append(fig(FIG_DIR / "training_sample_batch.png", width=17*cm))

doc.build(story)
print(f"Wrote {OUT_PDF}  ({OUT_PDF.stat().st_size/1024:.1f} KB)")


## Done

Artifacts produced:
- `other_files/checkpoints/{small,medium,large}.pt` -- best PyTorch checkpoints
- `other_files/training_report.json` -- machine-readable
- `other_files/training_report.pdf` -- human-readable
- `other_files/figs_eda/training_*.png` -- four figures

**Re-running this notebook is safe**: trained variants are auto-loaded from their `.pt` checkpoints. Set `FORCE_RETRAIN=True` to wipe and re-train.

**Next phase:** Phase 6 -- evaluation on the full test set with ablations (radar subset, clutter on/off, temporal context length), then convert the trained models to INT8 TFLite via `convert_to_litert.convert_int8()` for the final submission.
